# tiny-rlhf: train on a Colab T4

This notebook clones the private repo, runs one training config on the GPU, makes the training figure, and pushes the results back to the repo. Every number that ends up in the README comes from a script in the repo and is saved under `results/`.

Before running, read `docs/COLAB.md` in the repo. You need two things set up once:

1. A T4 GPU runtime (Runtime > Change runtime type > T4 GPU).
2. A secret named `GH_TOKEN` in the Secrets panel (key icon in the left sidebar) with notebook access switched on. `HF_TOKEN` is optional.

Run the cells from top to bottom.

## 1. Check the GPU

The run stops here if there is no GPU, so no time is wasted on a CPU runtime.

In [ ]:
import subprocess

import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU found. Pick Runtime > Change runtime type > T4 GPU, then run this cell again."
    )
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| device:", torch.cuda.get_device_name(0))

## 2. Parameters

These are the only values you normally change.

- `CONFIG`: which run to do. Start with `configs/dpo_tiny.yaml` as a short dry run that exercises the whole pipeline, then switch to `configs/dpo_full.yaml`.
- `RUN_NAME`: leave as `None` to use the `run_name` written in the config file. Set a string to override it.
- `EXTRA_ARGS`: extra command line arguments passed to `scripts/train_dpo.py` unchanged.

In [ ]:
REPO = "Varkot-dev/tiny-rlhf"
BRANCH = "claude/charming-cray-20zxv9"
CONFIG = "configs/dpo_full.yaml"
RUN_NAME = None  # None means: use the run_name in the config file
EXTRA_ARGS = ""

## 3. Read the secrets and clone the repo

`GH_TOKEN` is required: a fine-grained GitHub token limited to this one repository with Contents set to Read and write. `HF_TOKEN` is optional and only matters if a model or dataset is gated.

The token is passed to git through environment variables that live only in this notebook's process. It is not printed and it is not written into the cloned repo's `.git/config`.

In [ ]:
import base64
import os

from google.colab import userdata

try:
    _gh_token = userdata.get("GH_TOKEN")
except Exception as err:  # SecretNotFoundError or NotebookAccessError
    raise SystemExit(
        "Could not read the GH_TOKEN secret. Open the Secrets panel (key icon in the left "
        "sidebar), add a secret named GH_TOKEN, and switch on 'Notebook access' for it. "
        f"({type(err).__name__})"
    ) from None
if not _gh_token:
    raise SystemExit("The GH_TOKEN secret is empty.")

# Give every git command in this session an auth header for github.com,
# without storing the token in any file.
_basic = base64.b64encode(f"x-access-token:{_gh_token}".encode()).decode()
os.environ["GIT_CONFIG_COUNT"] = "1"
os.environ["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
os.environ["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {_basic}"
os.environ["GIT_TERMINAL_PROMPT"] = "0"
del _gh_token, _basic

try:
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    _hf_token = None
if _hf_token:
    os.environ["HF_TOKEN"] = _hf_token
    print("HF_TOKEN: set")
else:
    print("HF_TOKEN: not set (fine unless the model or dataset is gated)")
del _hf_token
print("GH_TOKEN: set")

In [ ]:
import shutil
import subprocess

WORKDIR = "/content/tiny-rlhf"


def sh(cmd: str, cwd: str | None = None) -> None:
    """Run a shell command, show its output, and stop the notebook if it fails."""
    proc = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print((proc.stdout + proc.stderr).strip())
    if proc.returncode != 0:
        raise SystemExit(f"Command failed (exit {proc.returncode}): {cmd}")


if os.path.isdir(WORKDIR):
    shutil.rmtree(WORKDIR)  # start from a clean clone every time this cell runs
sh(f"git clone --branch {BRANCH} https://github.com/{REPO}.git {WORKDIR}")
os.chdir(WORKDIR)
sh("git log -1 --format='%h %s'")

## 4. Install the project

Colab already has PyTorch with GPU support, so it is not reinstalled. This installs the repo in editable mode with its training and plotting dependencies.

In [ ]:
%pip install -q -e ".[train,viz,dev]"

## 5. Run the tests

A quick check that the loss functions and the data pipeline behave on this machine before spending GPU time.

In [ ]:
sh("python -m pytest -q")

## 6. Train

Training prints its progress here. The first logged loss should be log 2, about 0.693, because the policy starts equal to the reference. If it is not, stop and report it.

Time and peak memory are measured by the run itself and written to `run_info.json`.

In [ ]:
import yaml

with open(CONFIG) as f:
    _cfg = yaml.safe_load(f) or {}
RUN = RUN_NAME or _cfg.get("run_name") or os.path.splitext(os.path.basename(CONFIG))[0]
RUN_DIR = f"results/{RUN}"

_cmd = f"python scripts/train_dpo.py --config {CONFIG}"
if RUN_NAME:
    _cmd += f" --run-name {RUN_NAME}"
if EXTRA_ARGS:
    _cmd += f" {EXTRA_ARGS}"
print("run name:", RUN)
print(_cmd)

In [ ]:
!{_cmd}

if not os.path.isfile(f"{RUN_DIR}/run_info.json"):
    raise SystemExit(
        f"Training did not finish: {RUN_DIR}/run_info.json is missing. Read the output above."
    )

## 7. Make the training figure

The figure is made by a script in the repo from the CSV files the run just wrote.

In [ ]:
import glob

from IPython.display import Image, display

sh(f"python scripts/plot_training.py --run {RUN_DIR}")
_pngs = sorted(set(glob.glob(f"assets/*{RUN}*.png") + glob.glob(f"{RUN_DIR}/*.png")))
if not _pngs:
    print("No figure found under assets/ or the run directory. Check the script output above.")
for _p in _pngs:
    print(_p)
    display(Image(filename=_p))

## 8. Look at the run

`run_info.json` holds the measured time and peak memory. The first and last rows of `eval.csv` show where the held-out metrics started and ended.

In [ ]:
import json

import pandas as pd

with open(f"{RUN_DIR}/run_info.json") as f:
    print(json.dumps(json.load(f), indent=2))

_eval = pd.read_csv(f"{RUN_DIR}/eval.csv")
display(_eval.iloc[[0, -1]] if len(_eval) > 1 else _eval)

## 9. Commit and push the results

This commits the run's directory under `results/` and any new figures under `assets/`, merges in anything that was pushed to the branch while the run was going, and pushes. The LoRA adapter under `checkpoints/` is not tracked in git.

In [ ]:
sh('git config user.name "Varshith Kotagiri"')
sh('git config user.email "Varkot-dev@users.noreply.github.com"')
sh(f"git add {RUN_DIR}")
if os.path.isdir("assets"):
    sh("git add assets")

if subprocess.run("git diff --cached --quiet", shell=True).returncode == 0:
    print("Nothing new to commit.")
else:
    _gpu = torch.cuda.get_device_name(0)
    sh(f'git commit -q -m "Add results for run {RUN} (Colab {_gpu})"')
    sh(f"git fetch origin {BRANCH}")
    sh("git merge --no-edit FETCH_HEAD")
    sh(f"git push origin HEAD:{BRANCH}")
    sh("git log -1 --format='pushed %h %s'")

## 10. Optional: keep the adapter on Google Drive

The trained LoRA adapter is saved under `checkpoints/` and disappears when this runtime is recycled. Uncomment the lines below to copy it to your Drive. Colab will ask for permission to access Drive.

In [ ]:
# from google.colab import drive
#
# drive.mount("/content/drive")
# _dst = f"/content/drive/MyDrive/tiny-rlhf/checkpoints/{RUN}"
# shutil.copytree(f"checkpoints/{RUN}", _dst, dirs_exist_ok=True)
# print("copied to", _dst)